In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()


Successfully saved authorization token.


In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

### Method #1 for buffering the feature collection (did not work)

In [3]:
feature = ee.FeatureCollection("projects/deforestation-495419/assets/panama_protected_areas_polygons").filterBounds(panama_geom)

# Cast the resulting object as an ee.Feature so that the call to the buffer
# method is unambiguous (first() and buffer() are shared by multiple classes).
feature = ee.Feature(feature)

# Generate buffered features out and in from the original boundary.
buffer_out = feature.buffer(20000)  # 20 km out

### Method #2 for buffering the feature collection

In [4]:
features = ee.FeatureCollection("projects/deforestation-495419/assets/panama_protected_areas_polygons").filterBounds(panama_geom)

# Define the buffer function (Distance is in meters: 20km = 20000m) 
def add_buffer(feature):
    return feature.buffer(20000)

# Map the function over the FeatureCollection
buffered_features = features.map(add_buffer)

In [ ]:
# Method 1: Attribute Filtering (ISO3 Code) - looks at legally designated protected areas in Panama (therefore this one is better for our purposes)
panama_pas_iso = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

count_iso = panama_pas_iso.size().getInfo()
print(f"Count (via ISO3 code): {count_iso}")


# Method 2: Spatial Filtering (Using GAUL 2015 Boundary) - looks at whatever falls within the Panama boundary, regardless of legal designation
panama_boundary = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

panama_pas_spatial = ee.FeatureCollection("WCMC/WDPA/current/polygons").filterBounds(
    panama_boundary
)

count_spatial = panama_pas_spatial.size().getInfo()
print(f"Count (via Spatial Intersection): {count_spatial}")

Count (via ISO3 code): 78
Count (via Spatial Intersection): 87


### Quick List of Names

In [19]:
import ee

ee.Initialize()

# Load Panama protected areas
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Extract English names as a Python list
english_names = panama_pas.aggregate_array("NAME").getInfo()

# Print first 10 names
print(english_names)

['Serranía de Darién', 'Narganá', 'Alto Darién', 'Serranía del Bagre', 'Darién', 'Sistemas de Humedales de Matusagaratí', 'Chepigana', 'Isla del Rey', 'Majé', 'Filo del Tallo-Canglón', 'Darién', 'Punta Patiño', 'Tapagra', 'Parc national du Darien', 'Isla Bastimentos', 'Portobelo', 'Reverendo Padre Jesús Héctor Gallego Herrera', 'Cerro Hoya', 'Sarigua', 'Coiba', 'Humedal de Bahía de Panamá', 'Isla Boná', 'Isla de Cañas', 'Taboga-Urabá', 'Isla Iguana', 'Peñón de La Honda', 'La Playa de la Barqueta Agrícola', 'Manglares de Panamá Viejo', 'Zona de Reserva Matumbal', 'Playa Bluff', 'San San Pond Sak', 'Humedal de Bahía de Panamá', 'Golfo de Montijo', 'Parc national de Coiba et sa zone spéciale de protection marine', 'Golfo de Montijo', 'Escudo de Veraguas', 'Golfo de Chiriquí', 'Playa Boca Vieja', 'Isla Montuosa', 'Pablo Arturo Barrios', 'Zona de Reserva La Marinera', 'Cordillera de Coiba', 'Banco Volcán', 'Palo Seco', 'San Lorenzo', 'Laguna de Volcán', 'Los Pozos de Calobre', 'Cerro Gaital

### Extract WDPA IDs alongside Names and Polygons into a DataFrame
#### To match site IDs and polygon IDs directly with their names

In [22]:
import pandas as pd
import ee

ee.Initialize()

# Filter WDPA polygons for Panama
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Function to extract attributes cleanly per feature
def extract_props(feature):
    props = feature.toDictionary()
    return ee.Feature(None, {
        'WDPAID': props.get('WDPAID', 'N/A'),
        'WDPA_PID': props.get('WDPA_PID', 'N/A'),
        'ORIG_NAME': props.get('ORIG_NAME', 'N/A'),
        'NAME': props.get('NAME', 'N/A'),
        'DESIG_ENG': props.get('DESIG_ENG', 'N/A')
    })

# Extract property dictionaries
dict_list = panama_pas.map(extract_props).reduceColumns(
    ee.Reducer.toList(), ['system:index'] # dummy reducer to fetch feature list
).get('list') # Returns a list of dictionaries

# Convert directly to DataFrame
features_info = panama_pas.map(extract_props).getInfo()['features']
rows = [f['properties'] for f in features_info]

df = pd.DataFrame(rows)

# Reorder columns
df = df[['WDPAID', 'WDPA_PID', 'ORIG_NAME', 'NAME', 'DESIG_ENG']]

# print(df.head(10))
print(df)

   WDPAID WDPA_PID ORIG_NAME                                  NAME  \
0     N/A      N/A       N/A                    Serranía de Darién   
1     N/A      N/A       N/A                               Narganá   
2     N/A      N/A       N/A                           Alto Darién   
3     N/A      N/A       N/A                    Serranía del Bagre   
4     N/A      N/A       N/A                                Darién   
..    ...      ...       ...                                   ...   
73    N/A      N/A       N/A        Ribera Oeste del Lago Alajuela   
74    N/A      N/A       N/A                Ciénaga de Las Macanas   
75    N/A      N/A       N/A                        Bahía de Chame   
76    N/A      N/A       N/A                                Donoso   
77    N/A      N/A       N/A  Reserva de la Biósfera de La Amistad   

                       DESIG_ENG  
0           Hydrological Reserve  
1                      Wild Area  
2              Protective Forest  
3            Biolog

In [24]:
df

,WDPAID,WDPA_PID,ORIG_NAME,NAME,DESIG_ENG
0,N/A,N/A,N/A,Serranía de Darién,Hydrological Reserve
1,N/A,N/A,N/A,Narganá,Wild Area
2,N/A,N/A,N/A,Alto Darién,Protective Forest
3,N/A,N/A,N/A,Serranía del Bagre,Biological Corridor
4,N/A,N/A,N/A,Darién,National Park
...,...,...,...,...,...
73,N/A,N/A,N/A,Ribera Oeste del Lago Alajuela,Wild Area
74,N/A,N/A,N/A,Ciénaga de Las Macanas,Area of Managed Resources
75,N/A,N/A,N/A,Bahía de Chame,Multiple Use Areas
76,N/A,N/A,N/A,Donoso,Multiple Use Areas


### Provinces in each protected area

In [ ]:
# Because large protected areas (like Parque Nacional Chagres or Parque Nacional Darién) often cross province boundaries, 
# the code below attaches a list of all intersecting provinces to each record

import pandas as pd
import ee

ee.Initialize()

# Load Panama's Province Boundaries (GAUL Level 1)
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

# Load Panama's Protected Area Polygons (WDPA)
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Spatial Join Function: Find intersecting province(s) for each PA
def attach_province(pa):
    # Get provinces intersecting this PA geometry
    intersecting_provinces = provinces.filterBounds(pa.geometry())
    prov_list = intersecting_provinces.aggregate_array("ADM1_NAME")
    
    # Format province names into a comma-separated string
    # If empty (marine/offshore), assign "Marine / Offshore"
    prov_string = ee.Algorithms.If(
        prov_list.size().gt(0),
        prov_list.join(", "),
        "Marine / Offshore"
    )
    
    props = pa.toDictionary()
    return ee.Feature(None, {
        'WDPAID': props.get('WDPAID', 'N/A'),
        'WDPA_PID': props.get('WDPA_PID', 'N/A'),
        'ORIG_NAME': props.get('ORIG_NAME', 'N/A'),
        'NAME': props.get('NAME', 'N/A'),
        'DESIG_ENG': props.get('DESIG_ENG', 'N/A'),
        'PROVINCE': prov_string
    })

# Apply mapping and extract to Pandas
pa_with_provinces = panama_pas.map(attach_province)
features_info = pa_with_provinces.getInfo()['features']

rows = [f['properties'] for f in features_info]
province_df = pd.DataFrame(rows)

# Reorder columns
province_df = province_df[['WDPAID', 'WDPA_PID', 'ORIG_NAME', 'NAME', 'DESIG_ENG', 'PROVINCE']]

print(province_df)

# Optional: Save to CSV
# province_df.to_csv("panama_protected_areas_by_province.csv", index=False)

   WDPAID WDPA_PID ORIG_NAME                                  NAME  \
0     N/A      N/A       N/A                    Serranía de Darién   
1     N/A      N/A       N/A                               Narganá   
2     N/A      N/A       N/A                           Alto Darién   
3     N/A      N/A       N/A                    Serranía del Bagre   
4     N/A      N/A       N/A                                Darién   
..    ...      ...       ...                                   ...   
73    N/A      N/A       N/A        Ribera Oeste del Lago Alajuela   
74    N/A      N/A       N/A                Ciénaga de Las Macanas   
75    N/A      N/A       N/A                        Bahía de Chame   
76    N/A      N/A       N/A                                Donoso   
77    N/A      N/A       N/A  Reserva de la Biósfera de La Amistad   

                       DESIG_ENG                               PROVINCE  
0           Hydrological Reserve                      Darién, Kuna Yala  
1          

In [28]:
province_df

,WDPAID,WDPA_PID,ORIG_NAME,NAME,DESIG_ENG,PROVINCE
0,N/A,N/A,N/A,Serranía de Darién,Hydrological Reserve,"Darién, Kuna Yala"
1,N/A,N/A,N/A,Narganá,Wild Area,"Colón, Kuna Yala, Panamá"
2,N/A,N/A,N/A,Alto Darién,Protective Forest,"Darién, Emberá, Kuna Yala"
3,N/A,N/A,N/A,Serranía del Bagre,Biological Corridor,"Darién, Emberá"
4,N/A,N/A,N/A,Darién,National Park,"Darién, Emberá"
...,...,...,...,...,...,...
73,N/A,N/A,N/A,Ribera Oeste del Lago Alajuela,Wild Area,Colón
74,N/A,N/A,N/A,Ciénaga de Las Macanas,Area of Managed Resources,"Coclé, Herrera"
75,N/A,N/A,N/A,Bahía de Chame,Multiple Use Areas,Panamá
76,N/A,N/A,N/A,Donoso,Multiple Use Areas,"Veraguas, Coclé, Colón"


### Now with PA areas

#### GIS_AREA_KM2 is the exact geometrical area of the specific polygon slice in Earth Engine
#### REP_AREA_KM2 is the official area reported by the managing government agency. 
#### These can differ slightly due to spatial digitization errors, buffer zones, or maritime boundaries not fully captured in land shapefiles

In [27]:
import pandas as pd
import ee

ee.Initialize()

# 1. Load Panama Provinces and WDPA Polygons
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# 2. Function to compute geometry area and attach metadata + province
def process_protected_area(pa):
    # Compute GIS surface area in square kilometers
    area_sqkm = pa.geometry().area().divide(1e6)
    
    # Get intersecting province(s)
    intersecting_provinces = provinces.filterBounds(pa.geometry())
    prov_list = intersecting_provinces.aggregate_array("ADM1_NAME")
    
    prov_string = ee.Algorithms.If(
        prov_list.size().gt(0),
        prov_list.join(", "),
        "Marine / Offshore"
    )
    
    props = pa.toDictionary()
    
    return ee.Feature(None, {
        'WDPAID': props.get('WDPAID', 'N/A'),
        'WDPA_PID': props.get('WDPA_PID', 'N/A'),
        'ORIG_NAME': props.get('ORIG_NAME', 'N/A'),
        'NAME': props.get('NAME', 'N/A'),
        'PROVINCE': prov_string,
        'GIS_AREA_KM2': area_sqkm,
        'REP_AREA_KM2': props.get('REP_AREA', 'N/A')  # Reported area in WDPA
    })

# 3. Map over features and export to Pandas DataFrame
processed_pas = panama_pas.map(process_protected_area)
features_info = processed_pas.getInfo()['features']

rows = [f['properties'] for f in features_info]
df = pd.DataFrame(rows)

# Round surface area for readability
df['GIS_AREA_KM2'] = df['GIS_AREA_KM2'].round(2)

# Reorder columns
df = df[['WDPAID', 'WDPA_PID', 'ORIG_NAME', 'NAME', 'PROVINCE', 'GIS_AREA_KM2', 'REP_AREA_KM2']]

# Sort by largest surface area
df = df.sort_values(by='GIS_AREA_KM2', ascending=False).reset_index(drop=True)

print(df.head(15))

   WDPAID WDPA_PID ORIG_NAME  \
0     N/A      N/A       N/A   
1     N/A      N/A       N/A   
2     N/A      N/A       N/A   
3     N/A      N/A       N/A   
4     N/A      N/A       N/A   
5     N/A      N/A       N/A   
6     N/A      N/A       N/A   
7     N/A      N/A       N/A   
8     N/A      N/A       N/A   
9     N/A      N/A       N/A   
10    N/A      N/A       N/A   
11    N/A      N/A       N/A   
12    N/A      N/A       N/A   
13    N/A      N/A       N/A   
14    N/A      N/A       N/A   

                                                 NAME  \
0                                 Cordillera de Coiba   
1                                        Banco Volcán   
2                                              Darién   
3                                              Darién   
4                             Parc national du Darien   
5   Parc national de Coiba et sa zone spéciale de ...   
6                                               Coiba   
7                              

In [29]:
df

,WDPAID,WDPA_PID,ORIG_NAME,NAME,PROVINCE,GIS_AREA_KM2,REP_AREA_KM2
0,N/A,N/A,N/A,Cordillera de Coiba,Marine / Offshore,68204.74,67908.978834
1,N/A,N/A,N/A,Banco Volcán,Marine / Offshore,14269.95,14201.134174
2,N/A,N/A,N/A,Darién,"Darién, Emberá",5699.46,5689.578508
3,N/A,N/A,N/A,Darién,"Darién, Emberá",5699.46,5689.578508
4,N/A,N/A,N/A,Parc national du Darien,"Darién, Emberá, Kuna Yala",5490.05,5790.000000
...,...,...,...,...,...,...,...
73,N/A,N/A,N/A,Isla Iguana,Los Santos,1.49,1.482343
74,N/A,N/A,N/A,Manglares de Panamá Viejo,Panamá,0.84,0.839527
75,N/A,N/A,N/A,Punta Bruja y Dejal,Panamá,0.75,0.748226
76,N/A,N/A,N/A,El Salto de Las Palmas,Veraguas,0.56,0.558028


### Dataframe with PA names, areas, and provinces

In [32]:
import pandas as pd
import ee

ee.Initialize()

# 1. Load Panama Provinces and WDPA Polygons
provinces = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# 2. Function to compute surface area & classify Main vs. Other Provinces
def process_protected_area(pa):
    pa_geom = pa.geometry()
    
    # Calculate total GIS surface area in km²
    total_area_km2 = pa_geom.area().divide(1e6)
    
    # Find all provinces intersecting this protected area
    intersecting_provinces = provinces.filterBounds(pa_geom)
    
    # Map over intersecting provinces to compute overlap area per province
    def get_overlap(prov):
        prov_geom = prov.geometry()
        overlap_area = pa_geom.intersection(prov_geom, ee.ErrorMargin(1)).area().divide(1e6)
        return ee.Feature(None, {
            'prov_name': prov.get('ADM1_NAME'),
            'overlap_area': overlap_area
        })
    
    overlap_fc = intersecting_provinces.map(get_overlap)
    
    # Sort provinces by overlap area (descending)
    sorted_overlaps = overlap_fc.sort('overlap_area', False)
    
    # Extract list of province names sorted by largest area overlap
    prov_names = sorted_overlaps.aggregate_array('prov_name')
    
    # Logic for Main vs Other Provinces
    # If no land intersection (offshore/marine)
    main_prov = ee.Algorithms.If(
        prov_names.size().gt(0),
        prov_names.get(0),
        "Marine / Offshore"
    )
    
    other_provs = ee.Algorithms.If(
        prov_names.size().gt(1),
        prov_names.slice(1).join(", "),
        "None"
    )
    
    props = pa.toDictionary()
    
    return ee.Feature(None, {
        'NAME': props.get('NAME', 'N/A'),
        'MAIN_PROVINCE': main_prov,
        'OTHER_PROVINCES': other_provs,
        'GIS_AREA_KM2': total_area_km2,
        'REP_AREA_KM2': props.get('REP_AREA', 'N/A')
    })

# 3. Map over features and export to Pandas DataFrame
processed_pas = panama_pas.map(process_protected_area)
features_info = processed_pas.getInfo()['features']

rows = [f['properties'] for f in features_info]
df = pd.DataFrame(rows)

# Round GIS surface area for display
df['GIS_AREA_KM2'] = df['GIS_AREA_KM2'].round(2)

# Reorder columns
df = df[['NAME', 'MAIN_PROVINCE', 'OTHER_PROVINCES', 'GIS_AREA_KM2', 'REP_AREA_KM2']]

# Sort by largest surface area
df = df.sort_values(by='GIS_AREA_KM2', ascending=False).reset_index(drop=True)

# Display sample results
print(df)

                         NAME      MAIN_PROVINCE    OTHER_PROVINCES  \
0         Cordillera de Coiba  Marine / Offshore               None   
1                Banco Volcán  Marine / Offshore               None   
2                      Darién             Darién             Emberá   
3                      Darién             Darién             Emberá   
4     Parc national du Darien             Darién  Emberá, Kuna Yala   
..                        ...                ...                ...   
73                Isla Iguana         Los Santos               None   
74  Manglares de Panamá Viejo             Panamá               None   
75        Punta Bruja y Dejal             Panamá               None   
76     El Salto de Las Palmas           Veraguas               None   
77   Zona de Reserva Matumbal     Bocas del Toro               None   

    GIS_AREA_KM2  REP_AREA_KM2  
0       68204.74  67908.978834  
1       14269.95  14201.134174  
2        5699.46   5689.578508  
3        5699.4

In [33]:
df

,NAME,MAIN_PROVINCE,OTHER_PROVINCES,GIS_AREA_KM2,REP_AREA_KM2
0,Cordillera de Coiba,Marine / Offshore,None,68204.74,67908.978834
1,Banco Volcán,Marine / Offshore,None,14269.95,14201.134174
2,Darién,Darién,Emberá,5699.46,5689.578508
3,Darién,Darién,Emberá,5699.46,5689.578508
4,Parc national du Darien,Darién,"Emberá, Kuna Yala",5490.05,5790.000000
...,...,...,...,...,...
73,Isla Iguana,Los Santos,None,1.49,1.482343
74,Manglares de Panamá Viejo,Panamá,None,0.84,0.839527
75,Punta Bruja y Dejal,Panamá,None,0.75,0.748226
76,El Salto de Las Palmas,Veraguas,None,0.56,0.558028


### With landuse classes

#### Original classes: 
#### [1] Mature mixed broadleaf forest [2] Secondary mixed broadleaf forest [3] Mangrove forest,
#### [4] Orey Forest [5] Cativo Forest [6] Raffia forest [7] Planted coniferous forest [8] Planted forest of broadleaf trees
#### [9] Stubble and shrub vegetation [10] Herbaceous vegetation [11] Low-lying, floodable vegetation 
#### [12] Rocky outcrop and bare earth [13] Beach and natural sand dune [14] Coffee [15] Citrus [16] Oil palm
#### [17] Plantain/banana [18] Another permanent crop [19] Rice [20] Sugar cane [21] Mixed horticulture
#### [22] Corn [23] Pineapple [24] Another annual crop [25] Heterogeneous area of ​​agricultural production
#### [26] Grass [27] Water surface [28] Populated area [29] Infrastructure [30] Mining [31] Aquaculture pond
#### [32] Saltworks [33] Albino

#### New classes:
#### [10] Mature forest [20] Secondary forest [30] Primary forest [40] crops [50] plantation, 
#### [60] other vegetation [70] other landuse [80] other forest


In [ ]:
import pandas as pd
import ee

ee.Initialize()

# Geometry and Datasets
panama_geom = ee.FeatureCollection("FAO/GAUL/2015/level0").filter(ee.Filter.eq("ADM0_NAME", "Panama"))

# Reclassified Land Cover Image
sinia_prism = ee.Image("projects/deforestation-panama/assets/landuse_2021").clip(panama_geom)

lu_index = [10,20,30,40, 50, 60, 70, 80]

VisParams = {
    "min": 10,
    "max": 80, 
    "palette": [
        '32a65e',  # [10] Mature forest
        '32a65e',  # [20] Secondary forest
        '1f8d49',  # [30] Primary forest
        '7dc975',  # [40] Crops
        '04381d',  # [50] Plantation
        '026975',  # [60] Other vegetation
        '000000',  # [70] Other landuse
        '7a6c00',  # [80] Other forest 
       ]
    }
        
    # A flat list of pixel values to replace
from_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]
    # A corresponding list of new values
to_list = [10, 20, 80, 80, 80, 80, 20, 20, 60, 60, 60, 70, 70, 50, 40, 50, 40, 40, 40, 50, 40, 40, 40, 40, 50, 60, 27, 28, 29, 70, 70, 70, 70]

    # for each forest age, mask the lulc of the year immediately preceding abandonment
lulc_aggregated = ee.Image()
remapped_band = sinia_prism.remap(from_list, to_list)

# Remapped image with single band
lulc_aggregated = sinia_prism.remap(from_list, to_list).rename('landuse')

# Map pixel values to string labels for readable DataFrame outputs
class_labels = {
    '10': 'Mature forest',
    '20': 'Secondary forest',
    '30': 'Primary forest',
    '40': 'Crops',
    '50': 'Plantation',
    '60': 'Other vegetation',
    '70': 'Other landuse',
    '80': 'Other forest',
    '27': 'Water surface',
    '28': 'Populated area',
    '29': 'Infrastructure'
}

# WDPA Protected Areas
panama_pas = ee.FeatureCollection("WCMC/WDPA/current/polygons").filter(
    ee.Filter.eq("ISO3", "PAN")
)

# Server-side function to evaluate land use classes per protected area
def extract_landuse(feature):
    # Calculate pixel frequency count inside feature (at native ~30m resolution)
    histogram = lulc_aggregated.reduceRegion(
        reducer=ee.Reducer.frequencyHistogram(),
        geometry=feature.geometry(),
        scale=30,
        maxPixels=1e9
    ).get('landuse')
    
    props = feature.toDictionary()
    
    return ee.Feature(None, {
        'WDPAID': props.get('WDPAID', 'N/A'),
        'WDPA_PID': props.get('WDPA_PID', 'N/A'),
        'ORIG_NAME': props.get('ORIG_NAME', 'N/A'),
        'NAME': props.get('NAME', 'N/A'),
        'HISTOGRAM': histogram
    })

# Extract features to Python
processed = panama_pas.map(extract_landuse)
features_info = processed.getInfo()['features']

# Process histograms locally in Python to extract Dominant and Other Land Uses
rows = []
for f in features_info:
    props = f['properties']
    counts = props.get('HISTOGRAM', {})
    
    if counts:
        # Sort class keys by pixel counts (descending)
        sorted_classes = sorted(counts.items(), key=lambda item: item[1], reverse=True)
        
        # Translate codes to human-readable names
        dominant_code = sorted_classes[0][0]
        dominant_label = class_labels.get(dominant_code, f"Class {dominant_code}")
        
        other_labels = [
            class_labels.get(code, f"Class {code}") 
            for code, count in sorted_classes[1:]
        ]
        other_str = ", ".join(other_labels) if other_labels else "None"
    else:
        dominant_label = "No Data / Offshore"
        other_str = "None"
    
    rows.append({
        'NAME': props.get('NAME'),
        'DOMINANT_LANDUSE': dominant_label,
        'OTHER_LANDUSES': other_str
    })

# 5. Build Pandas DataFrame
df = pd.DataFrame(rows)
df = df[['NAME', 'DOMINANT_LANDUSE', 'OTHER_LANDUSES']]

print(df.head(15))

In [ ]:
Map = geemap.Map()
Map.centerObject(panama_geom, 7)

# Map.addLayer(sinia_prism, visParams, "SINIA Landuse")
Map.addLayer(remapped_band, VisParams, "Landuse Aggregated")

Map

In [5]:
Map.addLayer(features, {}, 'protected areas')
Map.addLayer(buffered_features, {}, '20km buffer')

Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…